# 05 — Recipe control, 12 seeds (OPTIONAL, EXPLORATORY)

## 1. Verify the GPU(s)

In [ ]:
!nvidia-smi

## 2. Clone the repo

Set `PROBE_REPO_URL` to your public Git URL or `PROBE_REPO_DIR` to a local checkout.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_DIR = os.environ.get("PROBE_REPO_DIR", "/kaggle/working/probe-capacity-invariance")
if not (Path(REPO_DIR) / "pyproject.toml").is_file():
    repo_url = os.environ.get("PROBE_REPO_URL")
    if not repo_url:
        raise RuntimeError("Set PROBE_REPO_URL or PROBE_REPO_DIR before running this notebook.")
    subprocess.run(["git", "clone", repo_url, REPO_DIR], check=True)
os.chdir(REPO_DIR)


## 3. Install dependencies

In [ ]:
!pip install -q -e . --no-deps
!pip install -q h5py

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "| device count:", torch.cuda.device_count())

## 4. Download shapes3d + build the image cache

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.data.shapes3d --download --build-cache

## 5. Restore checkpoints from a previous session (resume)

In [ ]:
import re, shutil
from pathlib import Path

REPO  = Path("/kaggle/working/probe-capacity-invariance")
ENC   = REPO / "results" / "encoders"
INPUT = Path("/kaggle/input")
_RID  = re.compile(r"scale_recipe_strong_seed\d+")

def _target(name):
    n = name.lower()
    if "ckpt" in n:     return "last_ckpt.pt"
    if "backbone" in n: return "backbone.pt"
    return None

found = {}
for p in sorted(INPUT.rglob("*.pt")) if INPUT.exists() else []:
    m, tgt = _RID.search(p.as_posix()), _target(p.name)
    if m and tgt:
        found.setdefault((m.group(0), tgt), p)

if not found:
    print("nothing to restore -- fresh start")
for (rid, tgt), src in sorted(found.items()):
    dst = ENC / rid / tgt
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists():
        shutil.copy2(src, dst)
    print(f"{rid:34s} {tgt:14s} <- {src}")

## 6. Integrity check — keep good checkpoints, purge corrupt ones

In [ ]:
import zipfile
from pathlib import Path

ENC = Path("/kaggle/working/probe-capacity-invariance/results/encoders")

def _state(p):
    if not p.exists():
        return "missing"
    try:
        return None if zipfile.ZipFile(p).testzip() is None else "corrupt"
    except Exception as e:
        return f"not-a-zip ({e})"

for d in sorted(ENC.glob("scale_recipe_strong_seed*")):
    bb, ck = d / "backbone.pt", d / "last_ckpt.pt"
    bstat = _state(bb)
    if bstat is None:
        print(f"{d.name:34s} backbone OK -> skip"); continue
    if bstat != "missing":
        bb.unlink(missing_ok=True); ck.unlink(missing_ok=True)
        print(f"{d.name:34s} backbone {bstat} -> purged, will retrain"); continue
    print(f"{d.name:34s} "
          + ("last_ckpt OK -> resume" if _state(ck) is None else "fresh start"))

## 7. Train seeds 0–11 (resumable)

In [ ]:










import torch
N_GPU = torch.cuda.device_count()
PER_GPU = 2 if N_GPU >= 2 else 3
print(f"{N_GPU} GPU(s) | per-gpu={PER_GPU} | {max(1, N_GPU) * PER_GPU} concurrent")

!cd /kaggle/working/probe-capacity-invariance && python -m src.encoders.launch_parallel \
    --config configs/run/scale_recipe_control.yaml \
    --seeds 0 1 2 3 4 5 6 7 8 9 10 11 \
    --per-gpu {PER_GPU} \
    --set data.in_memory=true run.num_workers=0


## 8. Probe the finished cell

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.probes.run_sweep \
    --config configs/probe/ladder.yaml \
    --dataset shapes3d --condition scale_recipe --strength strong \
    --encoders results/encoders/scale_recipe_strong_seed*/backbone.pt \
    --random-seed 0 1 2 3 4 5 6 7 8 9 10 11 --subsample 40000 \
    --device cuda --num-workers 2 --resume --out-root results/probes_excluded

## 9. Verdict — gate pass rate out of 12, classified per A8 §b

In [ ]:
import json
import numpy as np
from pathlib import Path

SHAPE_ABS_MIN = 0.90
d = Path("results/probes_excluded/scale_recipe_strong")
z = np.load(d / "stacks.npz"); m = json.loads((d / "meta.json").read_text())
names = [f["name"] for f in m["factors"]]
si = names.index("shape")

trained = z["trained"][:, si, :]
floor   = z["random"][:, si, :].mean(0)
lin = trained[:, 0]
n_pass = int((lin >= SHAPE_ABS_MIN).sum())
deficit = trained.mean(0) - floor

print(f"gate pass rate: {n_pass}/{len(lin)}  (threshold {SHAPE_ABS_MIN})")
print(f"linear-rung shape recoverability per seed: "
      f"{np.array2string(np.sort(lin), precision=3)}")
for r, rung in enumerate(m["rungs"]):
    print(f"  {rung:10s} trained {trained[:, r].mean():.3f}  floor {floor[r]:.3f}  "
          f"deficit {deficit[r]:+.3f}")

if n_pass >= 9:
    verdict = "PASS"
elif (deficit < 0).all() and deficit[-1] <= deficit[0]:
    verdict = "FAIL-DESTROYED"
else:
    verdict = "FAIL-INDETERMINATE"
print(f"\nA8 (b) CLASSIFICATION: {verdict}")
print({
 "PASS": "the colour-shortcut account survives: a photometric base recipe rescues the "
         "scale arm, so A5/A7 excluded their arms for the wrong reason. File an honest "
         "amendment saying so. EXPLORATORY - it does not re-enter the confirmatory grid, "
         "which A7 (d) froze at three cells.",
 "FAIL-DESTROYED": "shape information is destroyed, not made nonlinearly inaccessible, even "
         "with photometric augmentation. The construct-mismatch account stands and A7's "
         "exclusion keeps its gate licence.",
 "FAIL-INDETERMINATE": "a linear-readout collapse that closes with capacity. This does NOT "
         "license exclusion on gate grounds and does NOT settle colour-shortcut vs "
         "construct-mismatch: report both accounts as competing, per A7 (c).",
}[verdict])

In [ ]:



import shutil
from pathlib import Path
src = Path("/kaggle/working/probe-capacity-invariance/results"); dst = Path("/kaggle/working/results")
shutil.rmtree(dst, ignore_errors=True); shutil.copytree(src, dst)
print(f"persisted {sum(1 for _ in dst.rglob('*') if _.is_file())} files "
      f"-> click 'Save Version' now")